# Notebook 03 – Train the All-Star Prediction Pipeline

**What this notebook does:**
1. Loads `data/historical_features.parquet` built by notebook 01.
2. Performs a **season-based train/test split** (no random row shuffle).
3. Builds an **imblearn Pipeline**:
   - `ColumnTransformer` → numeric imputation + scaling, categorical OHE
   - `SMOTE` (fit on train only – no leakage)
   - `XGBClassifier`
4. Evaluates with PR-AUC, ROC-AUC, Precision@24, Recall@24.
5. Saves the fitted pipeline to `models/pipeline.joblib`.

**Prerequisites**: run `notebooks/01_build_dataset.ipynb` first.


In [45]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [46]:
# ── Configuration ────────────────────────────────────────────────────────
RANDOM_STATE = 42

DATA_PATH = REPO_ROOT / "data" / "preprocessed_features.csv"
MODELS_DIR  = REPO_ROOT / "models"
PIPELINE_PATH = MODELS_DIR / "pipeline.joblib"

MODELS_DIR.mkdir(parents=True, exist_ok=True)


In [47]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv(DATA_PATH)

df.head(3)


,PLAYER_AGE,GP,GS,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,...,team_PHI,team_PHO,team_POR,team_SAC,team_SAS,team_SEA,team_TOR,team_UTA,team_WAS,allstar
0,-0.986209,-0.140495,-0.812143,-0.889946,-0.684603,-0.714644,0.328438,-0.620109,-0.662199,-1.221719,...,-0.196516,-0.191317,5.338918,-0.180939,-0.193233,-0.140873,-0.163796,-0.184278,-0.19488,0
1,-0.741820,0.891807,-0.776443,-0.182671,0.001412,-0.061044,0.500859,-0.620109,-0.662199,-1.221719,...,-0.196516,-0.191317,5.338918,-0.180939,-0.193233,-0.140873,-0.163796,-0.184278,-0.19488,0
2,-0.497430,-1.283400,-0.812143,-1.033817,-0.846347,-0.874710,0.237691,-0.620109,-0.653565,-1.221719,...,-0.196516,-0.191317,-0.187304,-0.180939,-0.193233,-0.140873,-0.163796,-0.184278,-0.19488,0


## Season-based Train / Test Split

In [48]:
from sklearn.model_selection import train_test_split

# 02의 preprocessed_features.csv는 이미 one-hot encoded
# PLAYER_NAME, PLAYER_ID, year는 이미 제외됨
FEATURE_COLS = [c for c in df.columns if c != 'allstar']
NUMERIC_COLS = FEATURE_COLS  # 모든 컬럼이 numeric (team_*도 포함)
CAT_COLS = []  # 이미 one-hot encoded되었으므로 비움

X = df[FEATURE_COLS]
y = df['allstar']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train: {len(X_train)} rows, {int(y_train.sum())} All-Stars ({y_train.mean():.2%})")
print(f"Test : {len(X_test)} rows,  {int(y_test.sum())} All-Stars ({y_test.mean():.2%})")

Train: 17513 rows, 802 All-Stars (4.58%)
Test : 4379 rows,  200 All-Stars (4.57%)


## Build the Pipeline

We use `imblearn.pipeline.Pipeline` so that SMOTE is applied **only inside
the training fold** and never leaks into the test set.

```
ColumnTransformer
  ├─ numeric:      SimpleImputer(median) → StandardScaler
  └─ categorical:  OneHotEncoder(handle_unknown='ignore')
SMOTE(random_state=RANDOM_STATE)   ← train only
XGBClassifier(...)
```


In [56]:

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "logreg": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "rf": RandomForestClassifier(
        n_estimators=400,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced_subsample",
    ),
    "xgb": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

print("Model candidates ready.")

Model candidates ready.


## Evaluation

In [57]:
from src.evaluation import precision_at_k, recall_at_k
from sklearn.metrics import average_precision_score, roc_auc_score
import pandas as pd
import numpy as np

results_rows = []
fitted_pipelines = {}

for name, clf in models.items():
    pipeline = ImbPipeline(steps=[
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("classifier", clf),
    ])

    pipeline.fit(X_train, y_train)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    row = {
        "model": name,
        "P@24": precision_at_k(y_test.values, y_proba, k=24),
        "R@24": recall_at_k(y_test.values, y_proba, k=24),
        "PR-AUC": average_precision_score(y_test.values, y_proba),
        "ROC-AUC": roc_auc_score(y_test.values, y_proba) if len(np.unique(y_test.values)) > 1 else np.nan,
    }
    results_rows.append(row)
    fitted_pipelines[name] = pipeline
    print(f"[done] {name}")

results_df = pd.DataFrame(results_rows).sort_values(["PR-AUC", "P@24"], ascending=False)
print("\n=== Model comparison (test set) ===")
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]["model"]
pipeline = fitted_pipelines[best_model_name]
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(f"\nBest model: {best_model_name}")

[done] logreg
[done] rf
[done] xgb

=== Model comparison (test set) ===
 model     P@24  R@24   PR-AUC  ROC-AUC
logreg 0.875000 0.105 0.728436 0.982217
   xgb 0.833333 0.100 0.677670 0.978955
    rf 0.916667 0.110 0.653456 0.975853

Best model: logreg


## Save Pipeline Artifact

In [58]:
joblib.dump(pipeline, PIPELINE_PATH)
print(f"Pipeline saved to {PIPELINE_PATH}")

# Also save the feature column list for consistency checks during inference
import json
meta = {
    "best_model": best_model_name,  
    "feature_cols": FEATURE_COLS,
    "numeric_cols": NUMERIC_COLS,
    "cat_cols": CAT_COLS,
    "random_state": RANDOM_STATE,
}
with open(MODELS_DIR / "pipeline_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print("Metadata saved to models/pipeline_meta.json")

Pipeline saved to C:\Users\Dongyoon Nam\Desktop\NBA_ALLSTAR_PREDICTION\models\pipeline.joblib
Metadata saved to models/pipeline_meta.json
